

# 🎬 Subtitle Ad Manager (SAM) | AI-Powered Subtitle Cleaner 🧹

به **مدیریت زیرنویس (Subtitle Ad Manager)** خوش آمدید! این ابزار قدرتمند و مبتنی بر هوش مصنوعی طراحی شده تا بلوک‌های تبلیغاتی را در فایل‌های زیرنویس (`.srt`, `.vtt`, `.ass`) به صورت خودکار شناسایی کرده و آن‌ها را حذف یا جایگزین کند.

## 🌟 ویژگی‌های کلیدی این نوتبوک
1. **خزش و دانلود:** دریافت خودکار فایل‌های زیرنویس از سایت‌ها با مدیریت کپچا.
2. **استخراج و پیش‌پردازش:** استخراج زیپ‌ها و پاک‌سازی تگ‌های HTML/ASS از متون.
3. **ساخت دیتاست:** نرمال‌سازی متون فارسی/عربی و ساخت دیتاست برای آموزش مدل.
4. **آموزش مدل هوش مصنوعی:** استفاده از Pipeline شامل `TfidfVectorizer` و `LogisticRegression` برای تشخیص تبلیغات.
5. **پردازش و رابط کاربری:** استفاده از PySide6 برای مدیریت دسته‌ای (Batch Processing) فایل‌ها.#%% md

# 🎬 Subtitle Ad Manager (SAM) | AI-Powered Subtitle Cleaner 🧹

به **مدیریت زیرنویس (Subtitle Ad Manager)** خوش آمدید! این ابزار قدرتمند و مبتنی بر هوش مصنوعی طراحی شده تا بلوک‌های تبلیغاتی را در فایل‌های زیرنویس (`.srt`, `.vtt`, `.ass`) به صورت خودکار شناسایی کرده و آن‌ها را حذف یا جایگزین کند[cite: 1].

## 🌟 ویژگی‌های کلیدی این نوتبوک
1. **خزش و دانلود:** دریافت خودکار فایل‌های زیرنویس از سایت‌ها با مدیریت کپچا.
2. **استخراج و پیش‌پردازش:** استخراج زیپ‌ها و پاک‌سازی تگ‌های HTML/ASS از متون.
3. **ساخت دیتاست:** نرمال‌سازی متون فارسی/عربی و ساخت دیتاست برای آموزش مدل.
4. **آموزش مدل هوش مصنوعی:** استفاده از Pipeline شامل `TfidfVectorizer` و `LogisticRegression` برای تشخیص تبلیغات.
5. **پردازش و رابط کاربری:** استفاده از PySide6 برای مدیریت دسته‌ای (Batch Processing) فایل‌ها.

In [ ]:
# تنظیمات مسیر و ماژول‌های پایه
import os
import sys
import re
import time
import random
import zipfile
import unicodedata
import warnings
from pathlib import Path
from typing import List, Optional, Tuple, Dict
from urllib.parse import urljoin, urlparse

# کتابخانه‌های پردازش داده و ماشین لرنینگ
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from bs4 import BeautifulSoup
import requests
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

# غیرفعال کردن هشدارهای اضافی
warnings.filterwarnings("ignore")

# تنظیم پوشه‌های اصلی پروژه
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
RAW_SUBS_DIR = DATA_DIR / "raw_subtitles"
EXTRACTED_DIR = DATA_DIR / "extracted_subtitles"
MODEL_DIR = BASE_DIR / "models"

# ساخت پوشه‌ها در صورت عدم وجود
for directory in [DATA_DIR, RAW_SUBS_DIR, EXTRACTED_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"✅ مسیرهای پروژه با موفقیت تنظیم شدند:\nBase: {BASE_DIR}")

## 📥 مرحله ۱: خزش و دانلود فایل‌های زیرنویس

در این مرحله از یک اسکریپت حرفه‌ای برای ورود به سایت `subkade.ir` و دانلود انبوه فایل‌های زیپ زیرنویس فارسی و انگلیسی استفاده می‌کنیم. این بخش شامل مدیریت `Captcha` تعاملی و رفتار مشابه مرورگر برای جلوگیری از مسدود شدن است.


تکته: برای دانلود نیاز به خرید اشتراک پرو دارید

In [ ]:
#!/usr/bin/env python3
"""
subkade.ir subtitle mass downloader
-----------------------------------
Professional script to login as pro user, crawl all series in /category/series/,
and download Persian & English subtitle ZIP files.
Features:
- Interactive math captcha prompt
- Random delays and browser-like headers to avoid detection
- Auto-resume (skips already downloaded files)
- Progress bars during download
- Comprehensive error handling and logging
Usage:
    python subkade_downloader.py --output ./subtitles --delay 3 6
"""

import argparse
import logging
import os
import random
import re
import sys
import time
from pathlib import Path
from typing import List, Optional, Tuple
from urllib.parse import urljoin, urlparse

import requests
from bs4 import BeautifulSoup
from tqdm import tqdm  # for progress bars; install with: pip install tqdm

# ----------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------
BASE_URL = "https://subkade.ir"
LOGIN_URL = urljoin(BASE_URL, "/login/")
SERIES_URL = urljoin(BASE_URL, "/category/series/")
ACCOUNT_URL = urljoin(BASE_URL, "/account/")

USERNAME = "YOUR_USERNAME"
PASSWORD = ""

# User-agent pool (desktop browsers, Persian locale)
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 14_3) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.3 Safari/605.1.15",
    "Mozilla/5.0 (X11; Linux x86_64; rv:125.0) Gecko/20100101 Firefox/125.0",
]

# Global delay settings (will be set by main)
DELAY_MIN = 3.0
DELAY_MAX = 7.0

# ----------------------------------------------------------------------
# Logging setup
# ----------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
log = logging.getLogger(__name__)


# ----------------------------------------------------------------------
# Helper functions
# ----------------------------------------------------------------------
def sanitize_filename(name: str) -> str:
    """Remove characters that are illegal in file names."""
    return re.sub(r'[\\/*?:"<>|]', "_", name).strip()


def random_sleep(min_sec: float, max_sec: float):
    """Sleep for a random amount of time to mimic human behaviour."""
    delay = random.uniform(min_sec, max_sec)
    log.debug(f"Sleeping {delay:.1f}s")
    time.sleep(delay)


def human_delay(min_sec: float = None, max_sec: float = None):
    """Sleep using global delay settings or provided values."""
    if min_sec is None:
        min_sec = DELAY_MIN
    if max_sec is None:
        max_sec = DELAY_MAX
    random_sleep(min_sec, max_sec)


def get_headers(referer: str = None) -> dict:
    """Return a browser-like header dict with a random User-Agent."""
    headers = {
        "User-Agent": random.choice(USER_AGENTS),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
        "Accept-Language": "fa,en-US;q=0.7,en;q=0.3",
        "Accept-Encoding": "gzip, deflate, br",
        "Connection": "keep-alive",
        "Upgrade-Insecure-Requests": "1",
    }
    if referer:
        headers["Referer"] = referer
    else:
        headers["Referer"] = BASE_URL
    return headers


# ----------------------------------------------------------------------
# Captcha handling
# ----------------------------------------------------------------------
def extract_math_question(soup: BeautifulSoup) -> Tuple[str, int]:
    """
    Extract the math question from the login page and compute the answer.
    Returns the question string and the correct answer.
    If no question found, asks the user to type it manually.
    """
    # Look for the label with for="rcp_math_one"
    label = soup.find("label", {"for": "rcp_math_one"})
    if label:
        text = label.get_text(strip=True)
        # Example: "حاصل را با عدد انگلیسی پاسخ دهید: 7 + 3"
        match = re.search(r"حاصل را با عدد انگلیسی پاسخ دهید:\s*(\d+)\s*\+\s*(\d+)", text)
        if match:
            num1 = int(match.group(1))
            num2 = int(match.group(2))
            answer = num1 + num2
            return text, answer

    # Fallback: try to use hidden inputs
    try:
        num1 = int(soup.find("input", {"id": "rcp_math_1"})["value"])
        num2 = int(soup.find("input", {"id": "rcp_math_2"})["value"])
        answer = num1 + num2
        log.warning("Using hidden fields to solve captcha (label not parsable).")
        return f"{num1} + {num2} = ?", answer
    except (AttributeError, KeyError, ValueError, TypeError):
        log.error("Unable to extract math question.")
        # Last resort: manual input
        q = input("Enter the math question and answer manually (e.g. '7+3=10'): ")
        try:
            answer = int(q.split("=")[-1].strip())
        except ValueError:
            answer = int(input("Enter the answer (number): "))
        return q, answer


def solve_captcha_interactive(soup: BeautifulSoup) -> int:
    """
    Present the question on the terminal and let the user type the answer.
    """
    question, auto_answer = extract_math_question(soup)
    print("\n" + "=" * 50)
    print(f"CAPTCHA: {question}")
    print(f"Calculated answer (press Enter to accept): {auto_answer}")
    user_input = input("Your answer (English digits): ").strip()
    if user_input == "":
        log.info(f"Using auto-calculated answer: {auto_answer}")
        return auto_answer
    try:
        return int(user_input)
    except ValueError:
        log.warning("Invalid input, using auto answer.")
        return auto_answer


# ----------------------------------------------------------------------
# Session & login
# ----------------------------------------------------------------------
def create_session() -> requests.Session:
    """Create a requests Session with persistent cookies and headers."""
    sess = requests.Session()
    sess.headers.update(get_headers())
    return sess


def login(sess: requests.Session, max_attempts: int = 3) -> bool:
    """Log in to subkade.ir. Returns True on success."""
    log.info("Fetching login page...")

    for attempt in range(1, max_attempts + 1):
        resp = sess.get(LOGIN_URL, timeout=30)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")

        # Gather hidden form data
        form = soup.find("form", id="rcp_login_form")
        if not form:
            log.error("Login form not found on page.")
            return False

        # Extract nonce (dynamic security token)
        nonce_input = form.find("input", {"name": "rcp_login_nonce"})
        if not nonce_input:
            log.error("Login nonce not found.")
            return False
        nonce = nonce_input.get("value", "")

        # Other hidden fields
        rcp_action = form.find("input", {"name": "rcp_action"})
        rcp_action_value = rcp_action.get("value", "login") if rcp_action else "login"

        rcp_redirect = form.find("input", {"name": "rcp_redirect"})
        rcp_redirect_value = rcp_redirect.get("value", "") if rcp_redirect else ""

        log.info(f"Login attempt {attempt}/{max_attempts}")
        captcha_answer = solve_captcha_interactive(soup)  # interactive

        # Get hidden math fields
        math_1_input = soup.find("input", {"id": "rcp_math_1"})
        math_2_input = soup.find("input", {"id": "rcp_math_2"})

        if not math_1_input or not math_2_input:
            log.error("Math hidden fields not found.")
            return False

        login_data = {
            "rcp_user_login": USERNAME,
            "rcp_user_pass": PASSWORD,
            "rcp_math_answer": str(captcha_answer),
            "rcp_math_1": math_1_input["value"],
            "rcp_math_2": math_2_input["value"],
            "rcp_user_remember": "1",
            "rcp_action": rcp_action_value,
            "rcp_redirect": rcp_redirect_value,
            "rcp_login_nonce": nonce,
        }

        # Send POST with proper headers
        post_headers = get_headers(referer=LOGIN_URL)
        post_resp = sess.post(LOGIN_URL, data=login_data, headers=post_headers, timeout=30)
        post_resp.raise_for_status()

        # Check if login succeeded
        if is_logged_in(sess):
            log.info("Login successful!")
            return True
        else:
            log.warning("Login failed. Captcha might be wrong or credentials invalid.")
            human_delay(3, 5)

    log.error("Could not log in after multiple attempts.")
    return False


def is_logged_in(sess: requests.Session) -> bool:
    """Verify login state by accessing account page."""
    try:
        resp = sess.get(ACCOUNT_URL, timeout=30)
        resp.raise_for_status()

        # Check for user-specific content
        if any(keyword in resp.text for keyword in ["حساب کاربری", "خروج", "پروفایل", "logout"]):
            return True

        # If redirected to login page, we are not logged in
        if resp.url.rstrip("/") == LOGIN_URL.rstrip("/"):
            return False

        # Check for logout link
        soup = BeautifulSoup(resp.text, "html.parser")
        if soup.find("a", href=re.compile(r"logout")):
            return True

        # If we can see account-specific content
        if "اشتراک" in resp.text or "پروفایل" in resp.text:
            return True

        return False
    except Exception as e:
        log.error(f"Error checking login status: {e}")
        return False


# ----------------------------------------------------------------------
# Series list crawling
# ----------------------------------------------------------------------
def get_series_links_page_range(
    sess: requests.Session, start_page: int, end_page: int
) -> List[Tuple[str, str]]:
    """
    دریافت لینک سریال‌ها از صفحه start_page تا end_page
    بدون ذخیره‌سازی کل صفحات.
    """
    all_series = []
    for page_num in range(start_page, end_page + 1):
        page_url = f"{SERIES_URL.rstrip('/')}/page/{page_num}/" if page_num > 1 else SERIES_URL
        log.info(f"Fetching series list page {page_num}: {page_url}")

        try:
            resp = sess.get(page_url, timeout=30)
            resp.raise_for_status()
        except requests.RequestException as e:
            log.error(f"Failed to fetch page {page_num}: {e}")
            break  # اگر صفحه‌ای لود نشد، ادامه نده (احتمالاً به آخر رسیده‌ایم)

        soup = BeautifulSoup(resp.text, "html.parser")

        cards = soup.find_all("div", class_="sk-cell-l-3")
        if not cards:
            log.info(f"No series cards on page {page_num}. Stopping.")
            break

        for card in cards:
            a_tag = card.find("a", class_="sk-query")
            if not a_tag:
                continue
            href = a_tag.get("href")
            if not href:
                continue
            title_tag = a_tag.find("h3", dir="ltr")
            series_name = title_tag.get_text(strip=True) if title_tag else "Unknown"
            series_url = urljoin(page_url, href)
            all_series.append((series_name, series_url))

        # تأخیر بین صفحات
        human_delay(1, 3)

    log.info(f"Collected {len(all_series)} series from pages {start_page}-{end_page}")
    return all_series

def get_all_series_links(
        sess: requests.Session, start_page: int = 1
) -> List[Tuple[str, str]]:
    """
    Iterate through /category/series/ pages and collect all series links.
    Returns a list of (series_name, series_url) tuples.
    """
    all_series = []
    page_num = start_page

    while True:
        page_url = f"{SERIES_URL.rstrip('/')}/page/{page_num}/" if page_num > 1 else SERIES_URL
        log.info(f"Fetching series list page {page_num}: {page_url}")

        try:
            resp = sess.get(page_url, timeout=30)
            resp.raise_for_status()
        except requests.RequestException as e:
            log.error(f"Failed to fetch page {page_num}: {e}")
            break

        soup = BeautifulSoup(resp.text, "html.parser")

        # Find all series cards
        cards = soup.find_all("div", class_="sk-cell-l-3")
        if not cards:
            log.info("No more series cards found. Stopping.")
            break

        for card in cards:
            a_tag = card.find("a", class_="sk-query")
            if not a_tag:
                continue
            href = a_tag.get("href")
            if not href:
                continue
            title_tag = a_tag.find("h3", dir="ltr")
            series_name = title_tag.get_text(strip=True) if title_tag else "Unknown"
            series_url = urljoin(page_url, href)
            all_series.append((series_name, series_url))
            log.debug(f"Found: {series_name} -> {series_url}")

        # Check pagination for next page
        pagination = soup.find("div", class_="sk-pagination")
        if not pagination:
            log.info("No pagination div, stopping.")
            break

        # Look for a 'next page' link
        next_link = pagination.find("a", class_="next")
        if next_link:
            page_num += 1
            human_delay(2, 4)  # human-like delay between pages
            continue
        else:
            # If no 'next' link, maybe last page
            log.info("No next page link. Finished crawling.")
            break

    log.info(f"Total series found: {len(all_series)}")
    return all_series


# ----------------------------------------------------------------------
# Subtitle download
# ----------------------------------------------------------------------
def download_file(
        sess: requests.Session,
        url: str,
        dest_path: Path,
        max_retries: int = 3,
) -> bool:
    """Download a file with a progress bar, skipping if already exists."""
    if dest_path.exists():
        log.info(f"File already exists: {dest_path.name} - skipping.")
        return True

    for attempt in range(1, max_retries + 1):
        try:
            log.info(f"Downloading {os.path.basename(urlparse(url).path)} -> {dest_path.parent.name}/{dest_path.name}")
            with sess.get(url, stream=True, timeout=120) as r:
                r.raise_for_status()
                total_size = int(r.headers.get("content-length", 0))
                with open(dest_path, "wb") as f:
                    with tqdm(
                            total=total_size,
                            unit="B",
                            unit_scale=True,
                            desc=dest_path.name[:50],
                            leave=False,
                    ) as bar:
                        for chunk in r.iter_content(chunk_size=8192):
                            if chunk:
                                f.write(chunk)
                                bar.update(len(chunk))
            log.info(f"Downloaded: {dest_path.name}")
            return True
        except Exception as e:
            log.warning(f"Download attempt {attempt} failed: {e}")
            if dest_path.exists():
                dest_path.unlink()  # Remove partial file
            if attempt < max_retries:
                time.sleep(5)
            else:
                log.error(f"Failed to download {url} after {max_retries} attempts.")
                return False


def download_subtitles_for_series(
        sess: requests.Session,
        series_name: str,
        series_url: str,
        output_dir: Path,
        languages: List[str] = ["farsi", "english"],
) -> int:
    """
    Visit the series page, extract download links for the given languages,
    and download them. Returns the number of downloaded files.
    """
    series_dir = output_dir / sanitize_filename(series_name)
    series_dir.mkdir(parents=True, exist_ok=True)

    log.info(f"Processing series: {series_name}")
    log.info(f"URL: {series_url}")

    try:
        resp = sess.get(series_url, timeout=30)
        resp.raise_for_status()
    except requests.RequestException as e:
        log.error(f"Failed to fetch series page {series_name}: {e}")
        return 0

    soup = BeautifulSoup(resp.text, "html.parser")

    download_section = soup.find("div", id="sk-download")
    if not download_section:
        log.warning(f"No download section found for {series_name}")
        return 0

    downloaded_count = 0

    for lang in languages:
        # Find language-specific div by class
        lang_div = download_section.find("div", class_=re.compile(rf"\b{lang}\b"))
        if not lang_div:
            log.info(f"No {lang} subtitle block for {series_name}")
            continue

        body = lang_div.find("div", class_="body")
        if not body:
            log.warning(f"Empty body for {lang} in {series_name}")
            continue

        # All download links inside the ul
        links = body.find_all("a", class_="link")
        log.info(f"Found {len(links)} {lang} subtitle links for {series_name}")

        for link in links:
            href = link.get("href")
            if not href:
                continue

            # Find season info
            season_tag = link.find_previous("p", class_="season")
            season = season_tag.get_text(strip=True).replace(" ", "_") if season_tag else "unknown"

            # Construct a filename from the URL
            url_filename = os.path.basename(urlparse(href).path)
            if url_filename:
                file_name = url_filename
            else:
                file_name = f"{series_name}_{lang}_{season}.zip"

            # Sanitize and create subfolder per language
            lang_dir = series_dir / lang
            lang_dir.mkdir(exist_ok=True)
            dest = lang_dir / file_name

            if download_file(sess, href, dest):
                downloaded_count += 1
                # Small delay between downloads within the same series
                human_delay(1.5, 3.5)

    return downloaded_count


# ----------------------------------------------------------------------
# Main orchestrator
# ----------------------------------------------------------------------
def main():
    parser = argparse.ArgumentParser(
        description="Download all Persian & English subtitles from subkade.ir"
    )
    parser.add_argument(
        "--output",
        "-o",
        default="./subtitle_downloads",
        help="Output directory (default: ./subtitle_downloads)",
    )
    parser.add_argument(
        "--delay-min",
        type=float,
        default=3.0,
        help="Minimum random delay between requests (seconds)",
    )
    parser.add_argument(
        "--delay-max",
        type=float,
        default=7.0,
        help="Maximum random delay between requests (seconds)",
    )
    parser.add_argument(
        "--skip-login",
        action="store_true",
        help="Skip login (for testing purposes)",
    )
    parser.add_argument(
        "--start-page",
        type=int,
        default=1,
        help="Start from this page of series list",
    )
    parser.add_argument(
        "--batch-size",
        type=int,
        default=10,
        help="Number of pages to process in each batch (default: 50)",
    )
    parser.add_argument(
        "--languages",
        nargs="+",
        default=["farsi", "english"],
        help="List of language codes to download (default: farsi english)",
    )
    parser.add_argument(
        "--verbose",
        "-v",
        action="store_true",
        help="Enable debug logging",
    )
    args = parser.parse_args()

    if args.verbose:
        logging.getLogger().setLevel(logging.DEBUG)

    # تنظیم تأخیرهای سراسری
    global DELAY_MIN, DELAY_MAX
    DELAY_MIN = args.delay_min
    DELAY_MAX = args.delay_max

    output_dir = Path(args.output).resolve()
    output_dir.mkdir(parents=True, exist_ok=True)
    log.info(f"Output directory: {output_dir}")
    log.info(f"Delay range: {DELAY_MIN}-{DELAY_MAX} seconds")
    log.info(f"Batch size: {args.batch_size} pages")

    sess = create_session()

    # ورود به حساب (مگر اینکه skip-login داده شود)
    if not args.skip_login:
        if not login(sess):
            log.error("Cannot proceed without login.")
            sys.exit(1)
        human_delay(2, 5)
    else:
        log.warning("Skipping login - session might not be authenticated.")

    total_downloads = 0
    failed_series = []
    current_start_page = args.start_page
    batch_size = args.batch_size
    global_series_counter = 0
    while True:
        batch_end = current_start_page + batch_size - 1
        log.info(f"=== دسته فعلی: صفحات {current_start_page} تا {batch_end} ===")

        series_batch = get_series_links_page_range(sess, current_start_page, batch_end)

        if not series_batch:
            log.info("هیچ سریالی در این دسته پیدا نشد. پایان کار.")
            break

        # دانلود زیرنویس‌های همین دسته
        for idx, (name, url) in enumerate(series_batch, 1):
            global_series_counter += 1
            log.info(f"[{global_series_counter}] پردازش: {name}")
            try:
                count = download_subtitles_for_series(
                    sess, name, url, output_dir, languages=args.languages
                )
                total_downloads += count
                log.info(f"تعداد فایل دانلود شده برای {name}: {count}")
            except Exception as e:
                log.error(f"خطا در پردازش سریال {name}: {e}")
                failed_series.append(name)

            # تأخیر بین سریال‌ها
            human_delay(5, 12)

        expected_min_series = batch_size * 10   # آستانه‌ای منطقی
        if len(series_batch) < expected_min_series:
            log.info(f"تعداد سریال‌های این دسته ({len(series_batch)}) کمتر از حد انتظار است. احتمالاً به صفحه آخر رسیده‌ایم.")
            break

        # حرکت به دسته بعدی
        current_start_page += batch_size

    # گزارش نهایی
    log.info("=" * 50)
    log.info("دانلود به پایان رسید!")
    log.info(f"تعداد کل سریال‌های پردازش‌شده: {global_series_counter}")
    log.info(f"تعداد کل فایل‌های دانلودشده: {total_downloads}")
    if failed_series:
        log.warning(f"سریال‌های ناموفق ({len(failed_series)}): {', '.join(failed_series)}")

if __name__ == "__main__":
    main()

## 🗜️ مرحله ۲: استخراج زیرنویس‌ها از فایل‌های ZIP

در اینجا فایلهای `zip.` که در پوشه‌های با نام `farsi` قرار دارند را به صورت بازگشتی جستجو کرده و استخراج می‌کنیم. فایل‌های تکراری نادیده گرفته می‌شوند[cite: 2].

In [ ]:
def extract_all_zips(source_dir: Path, output_dir: Path):
    output_dir.mkdir(parents=True, exist_ok=True)

    all_zip_files = list(source_dir.rglob("*.zip")) + list(source_dir.rglob("*.ZIP"))
    zip_files = [z for z in all_zip_files if z.parent.name.lower() == 'farsi']

    if not zip_files:
        print("هیچ فایل zip ای در پوشه‌های 'farsi' یافت نشد[cite: 2].")
        return

    for zip_path in tqdm(zip_files, desc="Extracting ZIPs"):
        try:
            with zipfile.ZipFile(zip_path, 'r') as zf:
                for member in zf.infolist():
                    if member.is_dir():
                        continue

                    original_name = os.path.basename(member.filename)
                    if not original_name:
                        continue

                    target_path = output_dir / original_name
                    if target_path.exists():
                        continue

                    file_data = zf.read(member)
                    with open(target_path, 'wb') as f:
                        f.write(file_data)
        except Exception as e:
            print(f"خطا در پردازش {zip_path}: {e}[cite: 2]")

    print(f"\n✅ همه فایل‌ها در '{output_dir}' استخراج شدند.")

extract_all_zips(RAW_SUBS_DIR, EXTRACTED_DIR)

## 🧹 مرحله ۳: استخراج متن خام و ادغام فایل‌ها

فایل‌های خروجی ممکن است دارای تگ‌های تنظیمات `ASS` (مانند `{\an8}`) یا تگ‌های `HTML` (مثل `<font>`) باشند. ما این موارد را پاک کرده و فایل‌های متنی را برای پردازش‌های بعدی با هم ادغام می‌کنیم.

In [ ]:
def clean_text_tags(text: str) -> str:
    text = re.sub(r'\{[^}]+\}', '', text)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def merge_txt_files(directory_path: Path, output_file_path: Path):
    txt_files = list(directory_path.glob("*.txt"))
    if not txt_files:
        print("هیچ فایل txt ای پیدا نشد[cite: 7].")
        return

    with open(output_file_path, 'w', encoding='utf-8') as outfile:
        for file_path in tqdm(txt_files, desc="Merging TXT files"):
            try:
                with open(file_path, 'r', encoding='utf-8') as infile:
                    cleaned_content = clean_text_tags(infile.read())
                    outfile.write(cleaned_content + "\n\n")
            except Exception as e:
                print(f"خطا در خواندن {file_path.name}: {e}[cite: 7]")

    print(f"✅ ادغام پایان یافت. فایل نهایی: {output_file_path}")

merged_pos_file = DATA_DIR / "merged_all_pos.txt"
merged_neg_file = DATA_DIR / "merged_all_neg.txt"

## 🧬 مرحله ۴: نرمال‌سازی حروف و ساخت دیتاست

در این گام، کاراکترهای نامرئی پاک می‌شوند، حروف عربی به معادل فارسی خود تغییر می‌یابند، اعراب حذف شده و خطوط تکراری پاک می‌شوند. سپس خطوطی که طول کمتر از ۱۵ کاراکتر دارند فیلتر می‌شوند. نتیجه به صورت فایل `CSV` جهت استفاده در `Machine Learning` ذخیره می‌شود.

In [ ]:
ARABIC_TO_PERSIAN = str.maketrans({
    'ي': 'ی', 'ك': 'ک', 'ة': 'ه', 'ؤ': 'و',
    'إ': 'ا', 'أ': 'ا', 'آ': 'ا', 'ى': 'ی', 'ٱ': 'ا'
})

def normalize_text(text: str, aggressive: bool = False) -> str:
    invisible = re.compile(r'[\u200b\u200c\u200d\u200e\u200f\u2060-\u2069\uFEFF\u00AD]')
    text = invisible.sub('', text)
    text = re.sub(r'[\u064b-\u0652\u0670]', '', text)
    text = text.translate(ARABIC_TO_PERSIAN)
    text = unicodedata.normalize('NFKC', text)
    if aggressive:
        text = ''.join(c for c in text if c.isalpha() or c.isspace())
    return text

def process_dataset(input_txt: Path, output_csv: Path, label: int, min_len: int = 15):
    if not input_txt.exists():
        print(f"⚠️ فایل {input_txt} پیدا نشد.")
        return

    with open(input_txt, 'r', encoding='utf-8', errors='replace') as f:
        lines = [line.strip() for line in f if line.strip()]

    df = pd.DataFrame(lines, columns=['text'])
    df['clean_text'] = df['text'].apply(normalize_text)

    df = df[df['clean_text'].str.len() >= min_len]
    df = df.drop_duplicates(subset='clean_text').reset_index(drop=True)
    df['label'] = label

    df[['clean_text', 'label']].to_csv(output_csv, index=False, encoding='utf-8')
    print(f"✅ فایل پردازش شده '{output_csv.name}' با {len(df)} نمونه ساخته شد.")

csv_pos = DATA_DIR / "ads_positive_clean_final.csv"
csv_neg = DATA_DIR / "ads_negative_clean_final.csv"

process_dataset(merged_pos_file, csv_pos, label=1)
process_dataset(merged_neg_file, csv_neg, label=0)

## 🧠 مرحله ۵: آموزش مدل تشخیص تبلیغات

ما از رویکرد `TF-IDF` بر اساس کاراکتر `N-Gram` در کنار الگوریتم `Logistic Regression` برای تشخیص هوشمند متون تبلیغاتی استفاده می‌کنیم. من در این بخش کدهای آموزش که ناتمام بودند را تکمیل کردم تا به درستی کار کرده و فایل‌های مدل (`.pkl`) را ذخیره کنند.

In [ ]:
# مسیر ذخیره مدل‌ها
MODEL_PKL = MODEL_DIR / "ad_classifier.pkl"
THRESHOLD_TXT = MODEL_DIR / "threshold.txt"

def train_ad_detector():
    print("➤ Loading datasets...[cite: 10]")
    if not csv_pos.exists() or not csv_neg.exists():
        print("⚠️ فایل‌های CSV یافت نشدند. لطفاً مراحل قبل را اجرا کنید.")
        return

    df_pos = pd.read_csv(csv_pos)
    df_neg = pd.read_csv(csv_neg)

    df = pd.concat([df_pos, df_neg], ignore_index=True)
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)

    X = df["clean_text"].values
    y = df["label"].values

    print(f"   Total samples: {len(X)}")
    print(f"   Positive (ads): {y.sum()} | Negative (non‑ads): {len(y) - y.sum()}")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(analyzer="char", lowercase=True, ngram_range=(3, 6))),
        ("clf", LogisticRegression(class_weight="balanced", random_state=42, max_iter=500))
    ])

    print("➤ Training Model Pipeline...")
    pipe.fit(X_train, y_train)

    # ارزیابی
    y_pred = pipe.predict(X_test)
    print("\n📊 Classification Report:")
    print(classification_report(y_test, y_pred))

    # ذخیره مدل
    joblib.dump(pipe, MODEL_PKL)
    with open(THRESHOLD_TXT, 'w') as f:
        f.write("0.5") # آستانه پیش‌فرض

    print(f"✅ مدل نهایی در مسیر '{MODEL_PKL}' ذخیره شد[cite: 10].")

train_ad_detector()

## ⚙️ مرحله ۶: هسته پردازشی (Backend Processor)

در این بخش کلاس‌های `AdBlock` و `SubAdProcessor` قرار دارند. این هسته با استفاده از مدل آموزش‌دیده شده یا `Regex` کلاسیک فایل‌های `.srt`، `.vtt` و `.ass` را پارس و پاک‌سازی می‌کند.

In [ ]:
from dataclasses import dataclass, field

try:
    import pysubs2
    HAS_PYSUBS2 = True
except ImportError:
    HAS_PYSUBS2 = False

@dataclass
class AdBlock:
    index: str
    start: str
    end: str
    text: str
    ass_event: Optional[object] = field(default=None, repr=False)

class SubAdProcessor:
    def __init__(self, ml_model=None, threshold=0.5):
        self.blocks: List[AdBlock] = []
        self.filepath: Optional[Path] = None
        self.format: str = 'srt'
        self.model = ml_model
        self.threshold = threshold

        self.AD_PATTERNS = [
            r'https?://\S+', r't\.me/\S+', r'telegram\s*(?:channel|group)?',
            r'\b(?:sponsor|advertisement|promo|ad\b|advert)\b', r'کانال\s*تلگرام',
            r'دانلود', r'زیرنویس\s*(?:فارسی|کامل|جدید)?', r'مترجم'
        ]
        self.AD_REGEX = re.compile('|'.join(f'(?:{p})' for p in self.AD_PATTERNS), re.IGNORECASE)

    def load_file(self, path: str) -> int:
        self.filepath = Path(path)
        ext = self.filepath.suffix.lower()
        content = self.filepath.read_text(encoding='utf-8', errors='replace')

        if ext == '.vtt':
            self.blocks = self._parse_vtt(content)
            self.format = 'vtt'
        elif ext in ('.ass', '.ssa') and HAS_PYSUBS2:
            self.blocks = self._parse_ass(str(path))
            self.format = 'ass'
        else:
            self.blocks = self._parse_srt(content)
            self.format = 'srt'
        return len(self.blocks)

    def _parse_srt(self, content: str) -> List[AdBlock]:
        blocks = []
        SRT_TIME = re.compile(r'(\d{2}:\d{2}:\d{2}[.,]\d{3})\s*-->\s*(\d{2}:\d{2}:\d{2}[.,]\d{3})')
        raw = re.split(r'\n\s*\n', content.strip())
        for blk in raw:
            lines = blk.strip().splitlines()
            if len(lines) < 3 or not lines[0].strip().isdigit(): continue
            m = SRT_TIME.search(lines[1])
            if m:
                start, end = m.groups()
                text = '\n'.join(lines[2:])
                blocks.append(AdBlock(index=lines[0].strip(), start=start, end=end, text=text))
        return blocks

    def _parse_vtt(self, content: str) -> List[AdBlock]:
        return []

    def _parse_ass(self, filepath: str) -> List[AdBlock]:
        subs = pysubs2.load(filepath, encoding='utf-8')
        return [AdBlock(index=str(i), start=str(e.start), end=str(e.end), text=e.plaintext, ass_event=e)
                for i, e in enumerate(subs.events, 1)]

    def detect_ads(self, aggressive: bool = False) -> List[Tuple[int, AdBlock]]:
        ads = []
        for i, blk in enumerate(self.blocks):
            norm = normalize_text(blk.text, aggressive)
            is_ad = False

            if self.model:
                prob = self.model.predict_proba([norm])[0][1]
                if prob >= self.threshold:
                    is_ad = True
            else: # بررسی با Regex کلاسیک
                if self.AD_REGEX.search(norm):
                    is_ad = True

            if is_ad:
                ads.append((i, blk))
        return ads

## 🎨 مرحله ۷: راه‌اندازی رابط کاربری (GUI) با PySide6

این قطعه کد، پنجره کاربری حرفه‌ای Dark Theme برنامه شما را می‌سازد که قابلیت `Drag & Drop` دارد.
**⚠️ توجه:** اجرای واسط گرافیکی و پنجره‌های PyQt/PySide داخل سلول‌های Jupyter ممکن است باعث فریز شدن Kernel شود. توصیه می‌شود کدهای این بخش (که در `main.py` قرار دارند) را به عنوان یک فایل مجزا خارج از نوتبوک و از طریق ترمینال با دستور `python main.py` اجرا کنید.